In [1]:
import torch
print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

GPU: True
Device: Tesla T4


In [2]:
# !pip install datasets tokenizers tqdm tensorboard

In [2]:
# ── Environment setup ─────────────────────────────────────────────────────
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Directories are created by cfg.make_dirs() in the setup cell above

Project folders created.


In [5]:
from datasets import load_dataset
import config as cfg

dataset = load_dataset(cfg.PRETRAIN_DATASET_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [7]:
dataset["train"][0]

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

In [8]:
for i in range(3):
    print(dataset["train"][i]["text"])
    print("-----")

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
-----
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were fall

In [9]:
for i in range(3):
    print(dataset["validation"][i]["text"])
    print("-----")

Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.
-----
Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.

Roxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"

Roxy told Billy about the icy hill and how she couldn't climb it. Billy said, "I have an idea! Let's find som

In [24]:
with open(cfg.TRAIN_TXT, "w") as f:
    for item in dataset["train"]["text"]:
        text = item.replace("\n", " ")
        f.write(text + "\n")

print("Dataset saved to:", cfg.TRAIN_TXT)

Dataset saved to: /content/drive/MyDrive/slm_project/dataset/train.txt


In [25]:
num_val_samples = int(len(dataset["validation"]) * 0.5)
val_data  = dataset["validation"]["text"][:num_val_samples]
test_data = dataset["validation"]["text"][num_val_samples:]

with open(cfg.VAL_TXT, "w") as f:
    for item in val_data:
        text = item.replace("\n", " ")
        f.write(text + "\n")

with open(cfg.TEST_TXT, "w") as f:
    for item in test_data:
        text = item.replace("\n", " ")
        f.write(text + "\n")

print("Saved val →", cfg.VAL_TXT)
print("Saved test →", cfg.TEST_TXT)

Dataset saved to: /content/drive/MyDrive/slm_project/dataset/val.txt
Dataset saved to: /content/drive/MyDrive/slm_project/dataset/test.txt


In [ ]:
from tokenizer.preprocess import train_tokenizer
import config as cfg

tokenizer = train_tokenizer(cfg.TRAIN_TXT, cfg.TOKENIZER_DIR, cfg.VOCAB_SIZE)

Tokenizer saved.


In [ ]:
len(tokenizer.get_vocab())

8192

In [4]:
from tokenizer.preprocess import tokenize_and_save, load_tokenizer
import config as cfg

tokenizer = load_tokenizer(cfg.TOKENIZER_VOCAB, cfg.TOKENIZER_MERGES)

for split in ["train", "val", "test"]:
    tokenize_and_save(split, tokenizer, batch_size=4000)

Tokenizing /content/drive/MyDrive/slm_project/dataset/train.txt...


2119719it [18:27, 1913.60it/s]


Finished writing tokens to /content/drive/MyDrive/slm_project/dataset/train_tokens.npy
Tokenizing /content/drive/MyDrive/slm_project/dataset/val.txt...


10995it [00:04, 2587.45it/s]


Finished writing tokens to /content/drive/MyDrive/slm_project/dataset/val_tokens.npy
Tokenizing /content/drive/MyDrive/slm_project/dataset/test.txt...


10995it [00:04, 2304.19it/s]


Finished writing tokens to /content/drive/MyDrive/slm_project/dataset/test_tokens.npy


In [5]:
tokenizer.token_to_id("</s>")

2